In [3]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
import json
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
llm = ChatOllama(model="llama3.1", temperature=0)
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# --- 1. DEFINE THE TOOLS ---
@tool
def web_search(query: str) -> str:
    """Search the web for information. Use this to find facts, names, or dates."""
    print(f"🔎 [Tool Executing] Searching web for: {query}")
    # We are mocking the search result so it runs instantly on your machine
    if "ceo of microsoft" in query.lower():
        return "The CEO of Microsoft is Satya Nadella. He was born in the year 1967."
    return "Search result not found."

@tool
def calculate_age(birth_year: int, current_year: int) -> int:
    """Calculate an age given a birth year and the current year."""
    print(f"🧮 [Tool Executing] Calculating: {current_year} - {birth_year}")
    return current_year - birth_year

# Package them up and bind them to the LLM
my_tools = [web_search, calculate_age]
capstone_llm = llm.bind_tools(my_tools)


# --- 2. DEFINE THE NODES ---
def capstone_reasoning(state: AgentState):
    """The Brain."""
    response = capstone_llm.invoke(state['messages'])
    return {"messages": [response]}

def capstone_tools(state: AgentState):
    """The Hands (with guardrails)."""
    last_message = state['messages'][-1]
    
    if not getattr(last_message, 'tool_calls', None):
        return {"messages": []}

    tool_call = last_message.tool_calls[0]
    
    try:
        # Create a dictionary of our tools so the node knows which one to trigger
        available_tools = {"web_search": web_search, "calculate_age": calculate_age}
        chosen_tool = available_tools[tool_call["name"]]
        
        # Run the tool
        result = chosen_tool.invoke(tool_call["args"])
        return {"messages": [ToolMessage(content=str(result), tool_call_id=tool_call["id"])]}
        
    except Exception as e:
        # Our trusty Module 2 guardrail
        error_msg = f"Tool execution failed. Fix your format. Error: {str(e)}"
        print(f"⚠️ Guardrail triggered: {error_msg}")
        return {"messages": [ToolMessage(content=error_msg, tool_call_id=tool_call["id"])]}


# --- 3. DEFINE THE ROUTER ---
def capstone_router(state: AgentState):
    """The Traffic Cop."""
    last_message = state['messages'][-1]
    if getattr(last_message, 'tool_calls', None):
        return "tools"
    return "end"


# --- 4. BUILD THE GRAPH ---
builder = StateGraph(AgentState)
builder.add_node("brain", capstone_reasoning)
builder.add_node("hands", capstone_tools)

builder.add_edge(START, "brain")
builder.add_conditional_edges(
    "brain",
    capstone_router,
    {"tools": "hands", "end": END}
)
builder.add_edge("hands", "brain")

capstone_agent = builder.compile()

print("Module 5, Step 1 & 2 Complete: Capstone Agent is compiled and armed with two tools!")

Module 5, Step 1 & 2 Complete: Capstone Agent is compiled and armed with two tools!


In [4]:
# 1. The Multi-Step Prompt
# It needs a name/date from the web, and then it needs to do math with it.
query = "Find out who the CEO of Microsoft is, and then calculate how old they will be in the year 2030."

# 2. Initialize the clipboard
final_state = {"messages": [("user", query)]}

# 3. Set a slightly higher circuit breaker since we expect multiple loops
config = {"recursion_limit": 10}

print(f"👤 User: {query}\n")
print("=" * 50)

# 4. Stream the execution!
try:
    for event in capstone_agent.stream(final_state, config=config):
        for node_name, node_state in event.items():
            last_message = node_state['messages'][-1]
            
            # If the model called a tool...
            if getattr(last_message, 'tool_calls', None):
                tool_name = last_message.tool_calls[0]['name']
                tool_args = last_message.tool_calls[0]['args']
                print(f"🧠 [Brain] Decision: I need to use the '{tool_name}' tool.")
                print(f"   Arguments: {tool_args}")
            
            # If the tool just finished running...
            elif node_name == "hands":
                print(f"✅ [Hands] Result: {last_message.content}")
            
            # If the model is done and giving the final answer...
            else:
                print(f"\n🤖 [Brain] Final Answer: {last_message.content}")
                
            print("-" * 50)
            
except Exception as e:
    print(f"\n🚨 CIRCUIT BREAKER TRIPPED! The agent got confused. Error: {e}")

👤 User: Find out who the CEO of Microsoft is, and then calculate how old they will be in the year 2030.

🧠 [Brain] Decision: I need to use the 'web_search' tool.
   Arguments: {'query': 'Microsoft CEO'}
--------------------------------------------------
🔎 [Tool Executing] Searching web for: Microsoft CEO
✅ [Hands] Result: Search result not found.
--------------------------------------------------

🤖 [Brain] Final Answer: I'll try to find the current CEO of Microsoft.

{"name": "web_search", "parameters": {"query":"Microsoft CEO 2023"}}
{"name": "calculate_age", "parameters": {"birth_year":"1965","current_year":"2030"}}
--------------------------------------------------
